In [1]:
import numpy as np
import polars as pl
import altair as alt

In [2]:
np.random.seed(42)

## Mock Data

In [3]:
t1 = np.linspace(0, 4 * np.pi, 200)
series_a = np.sin(t1)

t2 = np.concatenate([
    np.linspace(0, 2 * np.pi, 120),
    np.linspace(2 * np.pi, 4 * np.pi, 80)
])
series_b = np.sin(t2)

series_a += np.random.normal(0, 0.03, len(series_a))
series_b += np.random.normal(0, 0.03, len(series_b))

In [4]:
wide_form = pl.DataFrame(
    {
        "x": np.arange(200),
        "A": series_a,
        "B": series_b,
    }
)
wide_form.head()

x,A,B
i64,f64,f64
0,0.014901,0.010734
1,0.058958,0.069599
2,0.14539,0.137895
3,0.234003,0.189352
4,0.242888,0.168303


In [5]:
long_form = pl.DataFrame({
    "x": np.concatenate([np.arange(len(series_a)),
                         np.arange(len(series_b))]),
    "y": np.concatenate([series_a, series_b]),
    "series": ["A"] * len(series_a) + ["B"] * len(series_b)
})
long_form.head()

x,y,series
i64,f64,str
0,0.014901,"""A"""
1,0.058958,"""A"""
2,0.14539,"""A"""
3,0.234003,"""A"""
4,0.242888,"""A"""


In [6]:
lines = (
    alt.Chart(long_form).mark_line(strokeWidth=2).encode(
    x=alt.X("x:Q", title="Time"),
    y=alt.Y("y:Q", title="Value"),
    color="series:N",
).properties(
    width=700, height=300, title="Two Similar Time Series with Temporal Warping"
))
lines

alt.Chart(...)

## Distance Calculation
### Euclidean Distance

In [7]:
euclidean_distance = np.sqrt(np.sum((series_a - series_b) ** 2))
euclidean_distance

np.float64(7.040543005810012)

In [8]:
sample_idx = np.arange(0, 200, 10)
rule_df = pl.DataFrame({
    "x": sample_idx,
    "y1": wide_form[sample_idx, "A"].to_list(),
    "y2": wide_form[sample_idx, "B"].to_list(),
})
rule_df.head()

x,y1,y2
i64,f64,f64
0,0.014901,0.010734
10,0.576434,0.520932
20,0.996958,0.939832
30,0.930036,0.978002
40,0.599675,0.833316


In [9]:
rules = (
    alt.Chart(rule_df)
    .mark_rule(color="red", strokeDash=[4, 4])
    .encode(x="x:Q", y="y1:Q", y2="y2:Q")
)
rules

alt.Chart(...)

In [10]:
chart = (lines + rules).properties(
    width=700,
    height=300,
    title="Euclidean Distance Forces Index-wise Alignment",
)
chart

alt.LayerChart(...)

## Find The Optimal Match

In [11]:
def dtw(series_a: np.ndarray, series_b: np.ndarray) -> dict[int, list[int]]:
    m, n = len(series_a), len(series_b)
    dp = [[float("inf") for _ in range(n + 1)] for _ in range (m + 1)]
    dp[0][0] = 0
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = min(
                dp[i - 1][j - 1],
                dp[i - 1][j],
                dp[i][j - 1],
            ) + abs(series_a[i - 1] - series_b[j - 1])

    # backtrace
    matches: dict[int, list[int]] = {}
    i, j = m, n
    while i > 0 and j > 0:
        matches.setdefault(i - 1, []).append(j - 1)
        candidates = [
            (dp[i - 1][j - 1], i - 1, j - 1),
            (dp[i][j - 1], i, j - 1),
            (dp[i - 1][j], i - 1, j),
        ]
        _, i, j = min(candidates, key=lambda t: t[0])

    return dp[-1][-1], matches

In [12]:
min_distance, optimal_matches = dtw(series_a, series_b)

In [13]:
min_distance

np.float64(5.126289792290117)

In [14]:
sample_idx

array([  0,  10,  20,  30,  40,  50,  60,  70,  80,  90, 100, 110, 120,
       130, 140, 150, 160, 170, 180, 190])

In [15]:
sample_idx_pair = [optimal_matches[i] for i in sample_idx]
sample_idx_pair

[[0],
 [11],
 [25],
 [33],
 [48, 47],
 [60],
 [72, 71],
 [83],
 [95, 94, 93],
 [108],
 [120, 119],
 [128],
 [137],
 [144],
 [152],
 [160],
 [168],
 [179],
 [185],
 [192]]

In [16]:
rows = []

for a_idx, b_indices in zip(sample_idx, sample_idx_pair):
    for b_idx in b_indices:
        gid = f"{a_idx}-{b_idx}"

        rows.append(
            {
                "x": a_idx,
                "y": series_a[a_idx],
                "group": gid,
            }
        )

        rows.append(
            {
                "x": b_idx,
                "y": series_b[b_idx],
                "group": gid,
            }
        )

edge_df = pl.DataFrame(rows)

In [17]:
edge_df

x,y,group
i64,f64,str
0,0.014901,"""0-0"""
0,0.010734,"""0-0"""
10,0.576434,"""10-11"""
11,0.582759,"""10-11"""
20,0.996958,"""20-25"""
…,…,…
179,-1.024051,"""170-179"""
180,-0.913198,"""180-185"""
185,-0.892727,"""180-185"""


In [18]:
edges = (
    alt.Chart(edge_df)
    .mark_line(color="red", strokeDash=[4, 4])
    .encode(
        x="x:Q",
        y="y:Q",
        detail="group:N",
    )
)
edges

alt.Chart(...)

In [19]:
chart = (lines + edges).properties(
    width=700,
    height=300,
    title="DTW Alignment Path",
)
chart

alt.LayerChart(...)